In [ ]:
%load_ext autoreload
%autoreload 2

# Must be set before numpy/scipy/statsmodels are imported (BLAS reads these at
# init). Multi-threaded MKL on this machine crashes the whole process outright
# (no exception, no crash dump) on any sizeable matrix op -- e.g. a single
# np.dot on a ~7.8M-row array, which statsmodels' Logit.predict() calls on
# every optimizer iteration. Forcing single-threaded BLAS fixed it completely
# (verified: the same np.dot that reliably killed the kernel went from
# crashing to running in 0.46s). Single-threaded is somewhat slower per BLAS
# call, but the actual matrix sizes here are small enough (~7.8M x ~130) that
# it's still sub-second either way.
import os
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'
os.environ['MKL_THREADING_LAYER'] = 'SEQUENTIAL'

import pipeline
import networkx as nx
import numpy as np
import pandas as pd
from pathlib import Path
import pickle
import sys
import re
import time

seed = 21
interval = 6
agg = True
vers = 'wgt'
weight = 'cov'

if agg:
    if interval == 6:
        fpath_pos = Path('data/agg/geotax_extended_6h_pos.csv.gz')
        fpath_neg = Path('data/agg/geotax_extended_6h_neg.csv.gz')
        save_dir = Path(f'data/agg/tests/link/6h/seed{seed}') if vers == 'link' else Path(f'data/agg/tests/wgt/{weight}/6h/seed{seed}')
    elif interval == 24:
        fpath = Path('data/agg/geotax_complete_24h.pkl')
        save_dir = Path(f'data/agg/tests/link/24h/seed{seed}') if vers == 'link' else Path(f'data/agg/tests/wgt/{weight}/24h/seed{seed}')
    else:
        print("Invalid time range (must be 24 or 6)")
else:
    if interval == 6:
        fpath = Path('data/poi_level/6h/thresholded_networks/0_network.txt')
        save_dir = Path(f'data/poi_level/6h/tests/seed{seed}')
        data = Path('data/poi_level/depnet_undirected.gz')
    elif interval == 24:
        fpath = Path('data/poi_level/24h/0_network_24h.txt')
        save_dir = Path(f'data/poi_level/24h/tests/seed{seed}')
        data = Path('data/poi_level/24h/depnet_undirected_24h.gz')
    else:
        print("Invalid time range (must be 24 or 6)")

save_dir.mkdir(parents=True, exist_ok=True)

def fast_read_csv(fpath, _dtypes):
    headers = pd.read_csv(fpath, nrows=0).columns
    _dtypes = {col: dt for col, dt in _dtypes.items() if col in headers}
    return pd.read_csv(fpath, dtype=_dtypes)

print(f"Interval: {interval} hrs | agg: {agg} | Seed: {seed} | Version: {vers}")

# Setup

In [ ]:
# set parameters

controlled = True
wgt_log = True
if weight:
    weighted = True
else:
    weighted = False
capweight = 'DEP' if weight == 'dep' else (
    'N_COVISITS' if weight == 'cov' else None)
operator = 'avg'

hyperparameters = {
    'p': 1,
    'q': 1,
    'workers': 6,
    'verbose': True,
    'dim': 128,
    'num_walks': 10,
    'walk_length': 80,
    'window_size': 10,
    'epochs': 4,
    'wgt_log': wgt_log
}

split_name = f'split_{weight}_log{wgt_log}_seed{seed}'
train_name = f'train_{weight}_log{wgt_log}_seed{seed}'

#### Reusable Split Function

In [ ]:
def new_split(data_path, split_name, train_name, vers, write=True, md_bool=False):

    split_name = save_dir / split_name
    train_name = save_dir / train_name if train_name else None

    if (write and not train_name) or (train_name and not write):
        cont_bool = input('Notice: "write" and "train_name" are incongruent. Continue? Y/N')
        if not (cont_bool == 'y' or cont_bool == 'Y'):
            print("Exiting...")
            sys.exit()

    split_name = Path(split_name)
    if split_name.is_file():
        cont_bool = input(f'Notice: split already exists for path {split_name}. Overwrite? Y/N')
        if not (cont_bool == 'y' or cont_bool == 'Y'):
            print("Exiting...")
            sys.exit()

    logistic = True
    if vers == 'wgt':
        logistic = False

    output = pipeline.prepare_data(
        data_path, logistic=logistic, test_frac=.25, seed=seed, weight=weight, metadata=md_bool,
        compress=wgt_log, write=write, trainfile=train_name)

    save_dir.mkdir(parents=True, exist_ok=True)
    with open(split_name, 'wb') as f:
        pickle.dump(output, f, pickle.HIGHEST_PROTOCOL)
    print(f"Saved split at {split_name}")

In [ ]:
# --- new split if needed ---

print(f'split name = {split_name}')
print(f'train name = {train_name}')

new_split(fpath_pos, split_name, train_name, vers=vers)

#### Outside Data

In [ ]:
# --- read in and save outside data if needed ---

# downcast for better performance
_dtypes = {
    'NODE_A': 'category', 'NODE_B': 'category',
    'N_COVISITS': 'float32', 'DIST_KM_MIN': 'float32',
    'DIST_KM_MEAN': 'float32', 'DIST_KM_MEDIAN': 'float32',
    'DIST_KM_CENTROID': 'float32', 'N_UIDS_A': 'float32',
    'N_POIS_A': 'float32', 'N_VISITS_A': 'float32',
    'N_UIDS_B': 'float32', 'N_POIS_B': 'float32',
    'N_VISITS_B': 'float32', 'DEP': 'float32'
}

df_pos = fast_read_csv(fpath_pos, _dtypes)
df_neg = fast_read_csv(fpath_neg, _dtypes)

# divide into train and test
def train_test_filter(df):
    train = df[df['SPLIT'] == 'TRAIN']
    test = df[df['SPLIT'] == 'TEST']
    return train, test

train_pos, test_pos = train_test_filter(df_pos)
train_neg, test_neg = train_test_filter(df_neg)

split_list = (train_pos, train_neg, test_pos, test_neg)
with open(save_dir / 'split_external_matched.pkl', 'wb') as f:
    pickle.dump(split_list, f)

In [ ]:
# --- write training graph from outside data if needed ---
if weight:
    print(f'wgt_log = {wgt_log}')
    G_train = nx.from_pandas_edgelist(
        train_pos, 'NODE_A', 'NODE_B', edge_attr=capweight)
    # rename to 'weight' for node2vec
    for u, v, data in G_train.edges(data=True):
        wgt_val = data.pop(capweight)
        if wgt_log:
            data['weight'] = np.log1p(wgt_val) if wgt_val > 0 else 0
        else:
            data['weight'] = wgt_val
    train_path = save_dir / f"train_external_matched_w{weight}_log{wgt_log}.txt"
else:
    G_train = nx.from_pandas_edgelist(train_pos, 'NODE_A', 'NODE_B')
    train_path = save_dir / "train_external_matched_wNone.txt"

if not train_path.is_file():
    with open(train_path, 'w') as f:
        for u, v, d in G_train.edges(data=True):
            f.write(f"{u}\t{v}\t{d.get('weight', 1.0)}\n")
        print(
            f"Wrote training graph: {G_train.number_of_nodes()} nodes, {G_train.number_of_edges()} edges")
else:
    print("Skipped. (File already exists)")

#### Reusable Test Run Function

In [ ]:
def run_sweep(
        feats_list, split_name, train_name, vers,
        reuse_file=None
    ):
    
    print(
        "Params check:\n"
        "============\n"
        f"controlled:{controlled}\n"
        f"weight:{weight} (weighted={weighted})\n"
        f"wgt_log:{wgt_log}\n"
        f"seed:{seed}\n"
        f"operator:{operator}\n"
    )

    green_light = input("Params ok? Y/N")
    if not (green_light == "y" or green_light == "Y"):
        print("Exiting...")
        sys.exit()

    # default values
    emb_map = None
    emb_store = []
    emb_bool = any('emb' in feats for feats in feats_list)

    if emb_bool:
        # sort into reuse/not
        if reuse_file:
            print("Reusing embeddings...\n")
            with open(save_dir / reuse_file, 'rb') as f:
                prev_results = pickle.load(f)
            emb_map = prev_results['embedding_map']
        else:
            print("Generating new embeddings...\n")
            emb_map = None
            emb_store = []
    
    def feats_to_txt(
        feats_list): return '+'.join(feats_list) if isinstance(feats_list, list) else feats_list

    # --- link branch ---

    if vers == 'link':
        # bring in split
        with open(save_dir / split_name, 'rb') as f:
                train_pos, train_neg, test_pos, test_neg = pickle.load(f)

        for feats in feats_list:
            feats_id = feats_to_txt(feats)
            print(f'NOW TESTING: {feats_id}\n============')
            if not reuse_file:
                # grab embedding map after first run
                if len(emb_store) == 1:
                    emb_map = emb_store[0]

            result = pipeline.run_pipeline_logistic(save_dir / train_name,
                                train_pos, train_neg, test_pos, test_neg, features=feats, seed=seed, **hyperparameters,
                                operator=operator, weighted=weighted, agg=agg, embedding_map=emb_map, dyadic_se=True)
            auc = result['auc']
            result['model'].remove_data()
            result = pd.Series(result)

            # save path handling

            if weight and emb_bool:
                out_file = save_dir / \
                    f'matching{controlled}/weight_{weight}/feats__{feats_id}/{operator}/wgtlog{wgt_log}'
            else:
                if weight and not emb_bool:
                    print("\nWeight irrelevant without embeddings. Saving to unweighted folder.")
                out_file = save_dir / \
                    f'matching{controlled}/weight_None/feats__{feats_id}/{operator}'
                
            if out_file.exists():
                run_numbers = [
                    int(match.group(1))
                    for d in out_file.iterdir()
                    if d.is_dir() and (match := re.fullmatch(r'run(\d+)', d.name))
                ]

                # compare against the params each earlier run was saved with
                target = None
                for n in run_numbers:
                    params_file = out_file / f'run{n}' / 'params.pkl'
                    if not params_file.is_file():
                        continue
                    with open(params_file, 'rb') as f:
                        old_params = pickle.load(f)
                    if old_params == hyperparameters:
                        overwrite_gate = input(f"run{n} already has identical params:\n{out_file / f'run{n}'}\nOverwrite? Y/N")
                        if not (overwrite_gate == 'y' or overwrite_gate == 'Y'):
                            print('\nAll tests completed.')
                            print("Exiting...")
                            sys.exit()
                        target = n
                        break

                run_idx = target if target is not None else max(run_numbers, default=0) + 1
                out_file = out_file / f'run{run_idx}'

            
            out_file.mkdir(parents=True, exist_ok=True)
    
            with open(out_file / 'auc_score.txt', 'w') as f:
                f.write(f'AUC: {auc:.6f}\n')
                f.write(f'==========\nParams: {hyperparameters}\n')
                print('Score saved.')
            with open(out_file / 'output.pkl', 'wb') as f:
                result.to_pickle(f)
                print('Output saved.\n')
            with open(out_file / 'params.pkl', 'wb') as f:
                pickle.dump(hyperparameters, f)
                print('Parameters saved.\n')
            if not reuse_file:
                if not emb_store:
                    emb_store.append(result['embedding_map'])
            del result

    # --- weight branch ---

    elif vers == 'wgt':
        with open(save_dir / split_name, 'rb') as f:
            split = pickle.load(f)
        train = split['train']
        test = split['test']

        for feats in feats_list:
            feats_id = feats_to_txt(feats)
            print(f'NOW TESTING: {feats_id}\n============')
            if not reuse_file:
                # grab embedding map after first run
                if len(emb_store) == 1:
                    emb_map = emb_store[0]

            result = pipeline.run_pipeline_linear(save_dir / train_name,
                                train, test, features=feats, seed=seed, **hyperparameters, compressed=wgt_log,
                                weight=weight, weighted=weighted, operator=operator, embedding_map=emb_map)
            result['model_results'].remove_data()
            result = pd.Series(result)

            # save path handling
            out_file = Path(save_dir / f'feats__{feats_id}/{operator}')
            if out_file.exists():
                run_numbers = [
                    int(match.group(1))
                    for d in out_file.iterdir()
                    if d.is_dir() and (match := re.fullmatch(r'run(\d+)', d.name))
                ]

                # compare against the params each earlier run was saved with
                target = None
                for n in run_numbers:
                    params_file = out_file / f'run{n}' / 'params.pkl'
                    if not params_file.is_file():
                        continue
                    with open(params_file, 'rb') as f:
                        old_params = pickle.load(f)
                    if old_params == hyperparameters:
                        overwrite_gate = input(f"run{n} already has identical params:\n{out_file / f'run{n}'}\nOverwrite? Y/N")
                        if not (overwrite_gate == 'y' or overwrite_gate == 'Y'):
                            print('\nAll tests completed.')
                            print("Exiting...")
                            sys.exit()
                        target = n
                        break

                run_idx = target if target is not None else max(run_numbers, default=0) + 1
                out_file = out_file / f'run{run_idx}'

            out_file.mkdir(parents=True, exist_ok=True)

            with open(out_file / 'output.pkl', 'wb') as f:
                result.to_pickle(f)
                print('Results saved.\n')
            with open(out_file / 'params.pkl', 'wb') as f:
                pickle.dump(hyperparameters, f)
                print('Parameters saved.\n')
            
            if not reuse_file:
                if not emb_store:
                    emb_store.append(result['embedding_map'])
            del result
    else:
        print(f'Unrecognized version: "{vers}"')
        return None

    print('\nAll tests completed.')

#### Reusable Hyperparameter Sweep Function

In [ ]:
def run_hparam_sweep(grids, split_name, train_name, vers):
    """
    Sequential node2vec hyperparameter sweep, emb as the only feature.

    Sweeps each param in `grids` one at a time (in dict order), holding the rest at
    their best value so far, starting from `hyperparameters`. Link runs are ranked on
    test AUC, weight runs on test R^2 (same ranking as RMSE on a fixed test set).

    Finished runs are saved to runs.pkl as they complete and reused instead of rerun,
    so an interrupted sweep resumes where it left off (delete runs.pkl to start over).

    Returns the full hyperparameter dict with the tuned values filled in, plus a dict
    of per-phase results dfs keyed by param.
    """
    if vers == 'link':
        metric, score_cols = 'auc', ['auc']
    elif vers == 'wgt':
        metric, score_cols = 'test_r2', ['rmse', 'mae', 'test_r2']
    else:
        print(f'Unrecognized version: "{vers}"')
        return None

    out_dir = save_dir / 'hparam_sweep' / Path(train_name).stem / operator
    runs_path = out_dir / 'runs.pkl'
    runs = pd.read_pickle(runs_path).to_dict('records') if runs_path.is_file() else []

    print(
        "Params check:\n"
        "============\n"
        f"vers:{vers} (ranked on {metric})\n"
        f"controlled:{controlled}\n"
        f"weight:{weight} (weighted={weighted})\n"
        f"wgt_log:{wgt_log}\n"
        f"seed:{seed}\n"
        f"operator:{operator}\n"
        f"split:{split_name} | train:{train_name}\n"
        f"start:{hyperparameters}\n"
        f"grids:{grids} (up to {sum(len(v) for v in grids.values())} runs)\n"
        f"saving to:{out_dir} ({len(runs)} runs already on file)\n"
    )

    green_light = input("Params ok? Y/N")
    if not (green_light == 'y' or green_light == 'Y'):
        print("Exiting...")
        sys.exit()

    out_dir.mkdir(parents=True, exist_ok=True)

    # bring in split
    with open(save_dir / split_name, 'rb') as f:
        if vers == 'link':
            train_pos, train_neg, test_pos, test_neg = pickle.load(f)
        else:
            split = pickle.load(f)
            train, test = split['train'], split['test']

    best = dict(hyperparameters)
    phases = {}

    for name, values in grids.items():
        fixed_str = ', '.join(f"{k}={best[k]}" for k in grids if k != name)
        print(f"\n{'='*60}")
        print(f"PHASE: sweeping {name}  |  fixed: {fixed_str}")
        print(f"  values: {values}")
        print(f"{'='*60}")

        rows = []
        for val in values:
            params = {**best, name: val}

            # workers/verbose don't change results, so they don't count toward a match
            row = next((r for r in runs if all(
                r.get(k) == v for k, v in params.items() if k not in ('workers', 'verbose'))), None)
            if row is not None:
                print(f"\n  [{name}={val}]  {metric} = {row[metric]:.4f}   (already run, reusing)")
                rows.append(row)
                continue

            print(f"\n  [{name}={val}] starting ...", flush=True)
            t0 = time.perf_counter()

            if vers == 'link':
                # no dyadic SEs: they don't change the AUC and only add compute
                result = pipeline.run_pipeline_logistic(save_dir / train_name,
                                    train_pos, train_neg, test_pos, test_neg, features=['emb'], seed=seed, **params,
                                    operator=operator, weighted=weighted, agg=agg)
                scores = {'auc': result['auc']}
            else:
                result = pipeline.run_pipeline_linear(save_dir / train_name,
                                    train, test, features=['emb'], seed=seed, **params,
                                    weight=weight, operator=operator, weighted=weighted, compressed=wgt_log)
                scores = {k: result['pred_results'][k] for k in score_cols}
            # drop the model + embeddings now, or they stay resident through the next run
            del result

            elapsed = time.perf_counter() - t0
            row = {**params, **scores, 'time_min': elapsed / 60}
            print(f"  [{name}={val}]  {metric} = {row[metric]:.4f}   ({elapsed/60:.1f} min)", flush=True)
            rows.append(row)

            # save after every run so a crash only loses the run in progress
            runs.append(row)
            pd.DataFrame(runs).to_pickle(runs_path)

        phase_df = pd.DataFrame(rows)
        best[name] = values[int(phase_df[metric].to_numpy().argmax())]
        phases[name] = phase_df
        print(f"\n  --> Best {name} = {best[name]}  ({metric} = {phase_df[metric].max():.4f})")
        if best[name] in (min(values), max(values)):
            print(f"  Notice: best {name} is at the edge of its grid. The optimum may lie beyond it.")
        print(phase_df[[name, *score_cols, 'time_min']].to_string(index=False))

    with open(out_dir / 'sweep_results.pkl', 'wb') as f:
        pickle.dump({'phases': phases, 'best': best}, f, pickle.HIGHEST_PROTOCOL)

    print(f"\n{'='*60}")
    print("SWEEP COMPLETE")
    for name in grids:
        print(f"  best {name} = {best[name]}")
    print(f"  all results -> {out_dir}")
    print(f"{'='*60}")

    return best, phases

# Run

### Link Prediction

In [ ]:
# --- hyperparameter sweep --- (emb only, run before the sweeps below)

if vers != 'link':
    print(f"Wrong version.")
    sys.exit()

split_path = 'split_external_matched.pkl'
train_path = f'train_external_matched_w{weight}_log{wgt_log}.txt' if weight else 'train_external_matched_wNone.txt'

hparam_grids = {
    'p': [0.25, 0.5, 1, 2, 4],
    'q': [0.25, 0.5, 1, 2, 4],
    'num_walks': [5, 10, 20],
    'walk_length': [40, 80, 160],
    'window_size': [5, 10, 20],
}

best_params, hparam_phases = run_hparam_sweep(hparam_grids, split_path, train_path, vers)

# carry the tuned values into the sweeps below
hyperparameters.update(best_params)

In [ ]:
# --- single run --- (curr: avg-cosine combo)

print(f"Interval: {interval} hrs | agg: {agg} | Seed: {seed} | 'Weight: {weight}")

feats = ['emb', 'cosine', 'dist_median']

result = pipeline.run_pipeline(
    save_dir / f"train_external_matched_w{weight}_log{wgt_log}.txt", train_pos, train_neg, 
    test_pos, test_neg, features=feats, seed=seed, **hyperparameters, 
    weighted=weighted, agg=agg, operator='avg'
)

link_auc = result['link_auc']
print(f"Link AUC: {link_auc:.4f}")
result['link_model'].remove_data()
output = pd.Series(result)
emb_map = result['embedding_map']

# save both score as txt and full output as pickle
feats_id = '__'.join(feats) if isinstance(feats, list) else feats
out_file = save_dir / f'matching{controlled}/weight_{weight}'
out_file.mkdir(parents=True, exist_ok=True)
with open(out_file / 'auc_scores.txt', 'w') as f:
    f.write(str(link_auc) + '\n')
    print('Score saved.')
with open(out_file / 'output.pkl', 'wb') as f:
    output.to_pickle(f)
    print('Output saved.')

In [ ]:
# --- sweep --- (curr: latlon)

if vers != 'link':
    print(f"Wrong version.")
    sys.exit()

split_path = 'split_external_matched.pkl'
if weight:
    train_path = f'train_external_matched_wcov_logTrue.txt'
else:
    train_path = 'train_external_matched_wNone.txt'

# latlon

latlon_sweep = [
    ['emb'],
    ['latlon'],
    ['emb', 'latlon']
]

run_sweep(latlon_sweep, split_path, train_path, vers)

### Weight Prediction

In [ ]:
if vers != 'wgt':
    print(f"Wrong version.")
    sys.exit()

print(f'split name = {split_name}')
print(f'train name = {train_name}')

# --- test 1: hyperparameter sweep ---

hparam_grids = {
    'p': [0.25, 0.5, 1, 2, 4],
    'q': [0.25, 0.5, 1, 2, 4],
    'num_walks': [5, 10, 20],
    'walk_length': [40, 80, 160],
    'window_size': [5, 10, 20],
}

best_params, hparam_phases = run_hparam_sweep(hparam_grids, split_name, train_name, vers)

# carry the tuned values into the tests below
hyperparameters.update(best_params)

# --- test 2: baseline + operator sweep ---

for op in pipeline.BINARY_OPERATORS.keys():
    operator=op
    run_sweep([['emb'], ['emb', 'dist_median']], split_name, train_name, vers)

### Embedding Location Probe

In [ ]:
# --- emb -> latlon regression --- (can a node's location be read off its embedding?)
# linear regression of each node's lat/lon on its 128-d embedding, one row per node, scored with
# 5-fold cross-validation. linear is a lower bound: a nonlinear model could decode more

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score

# emb-only runs whose embeddings were used in the latlon tests (tuned params). operator is
# hardcoded because the weight cell's operator loop leaves the global set to its last value
probe_runs = {
    'link (tuned)': Path(f'data/agg/tests/link/{interval}h/seed{seed}/matching{controlled}/weight_{weight}'
                         f'/feats__emb/avg/wgtlog{wgt_log}/run2'),
    'weight (tuned)': Path(f'data/agg/tests/wgt/{weight}/{interval}h/seed{seed}/feats__emb/avg/run1'),
}

# node locations (every node appears in the positives)
locs = pd.read_csv(fpath_pos, usecols=['NODE_A', 'NODE_B', 'LAT_A', 'LNG_A', 'LAT_B', 'LNG_B'])
locs = pd.concat([
    locs[['NODE_A', 'LAT_A', 'LNG_A']].set_axis(['node', 'lat', 'lng'], axis=1),
    locs[['NODE_B', 'LAT_B', 'LNG_B']].set_axis(['node', 'lat', 'lng'], axis=1),
]).drop_duplicates('node').set_index('node')

rows = []
for label, run_dir in probe_runs.items():
    emb_map = pd.read_pickle(run_dir / 'output.pkl')['embedding_map']
    with open(run_dir / 'params.pkl', 'rb') as f:
        params = pickle.load(f)

    nodes = list(emb_map.keys())
    missing = [n for n in nodes if n not in locs.index]
    if missing:
        print(f'Error: {len(missing)} embedded nodes have no location (e.g. {missing[:3]})')
        sys.exit()

    X = emb_map.rows(nodes)
    Y = locs.loc[nodes, ['lat', 'lng']].to_numpy()
    pred = cross_val_predict(LinearRegression(), X, Y, cv=KFold(5, shuffle=True, random_state=seed))

    # error as a distance on the map, vs just guessing the average location for every node
    err_km = pipeline.haversine(Y[:, 0], Y[:, 1], pred[:, 0], pred[:, 1])
    guess_km = pipeline.haversine(Y[:, 0], Y[:, 1], Y[:, 0].mean(), Y[:, 1].mean())

    rows.append({
        'embeddings': label,
        'params': f"p={params['p']} q={params['q']} nw={params['num_walks']} wl={params['walk_length']} ws={params['window_size']}",
        'nodes': len(nodes),
        'r2_lat': r2_score(Y[:, 0], pred[:, 0]),
        'r2_lng': r2_score(Y[:, 1], pred[:, 1]),
        'median_err_km': np.median(err_km),
        'p90_err_km': np.quantile(err_km, .9),
        'median_err_km_if_guessing_mean': np.median(guess_km),
    })

probe_df = pd.DataFrame(rows).set_index('embeddings')
print(probe_df.round(3).to_string())